# SaaS Release Impact Analysis

**Objective:** A SaaS company was experiencing a high volume of support tickets, driving up costs. Analysis revealed that many tickets were general product inquiries. In response, the company published a detailed product manual as a PDF download on their website (Release v1.2).

**My task:** Determine whether the introduction of the PDF download actually led to a reduction in support requests — and investigate any side effects such as backend errors.

*Note: To ensure an unbiased Exploratory Data Analysis, I created a synthetic dataset using Python (Faker module). The data structure was specifically designed to reflect real business scenarios of a SaaS company.*

---
## Part 1: Exploratory Data Analysis (Python)

*At the time of this analysis, data manipulation with Python had not yet been covered in my course. This section therefore focuses on an initial exploration of the data.*

### Loading the Data

In [ ]:
import pandas as pd

deployment_history = pd.read_csv("data/deployment_history_logs.csv")
raw_crm_customer_data = pd.read_csv("data/raw_crm_customer_data.csv")
sys_backend_error_logs = pd.read_csv("data/sys_backend_error_logs.csv")
web_tracking_event_log = pd.read_csv("data/web_tracking_event_log.csv")
zendesk_tickets_export = pd.read_csv("data/zendesk_tickets_export.csv")

### Deployment History

First, let's look at the release timeline to understand when the PDF download feature was deployed.

In [ ]:
print(deployment_history)

  version_id          deployed_at                           notes
0        1.0  2025-09-22 10:00:00                 Initial rollout
1        1.1  2025-10-06 10:00:00              Minor improvements
2        1.2  2025-10-20 10:00:00  Release v1.2 (PDF flow update)
3      1.2.1  2025-10-27 10:00:00                Hotfix unrelated


### CRM Customer Data

Exploring the customer data to understand which channels and operating systems are being used.

In [ ]:
show_row = raw_crm_customer_data.iloc[0].to_frame()
print(show_row)

                                                 0
user_id       caffcbdc-eabc-47a8-9204-4b98c7e9bdd7
created_at              2025-11-07 18:42:05.751570
country                                         AR
browser_type                                Chrome
device_type                                Desktop
os                                             iOS


This table links users to their browser, device and OS — useful for later analysis of error patterns.

### Backend Error Logs

Checking for error patterns in the backend.

In [ ]:
show_row = sys_backend_error_logs.iloc[0].to_frame()
print(show_row)

                                                    0
session_id       7065fb7d-6e74-491c-87c5-5f00bac01d45
error_timestamp                   2025-11-15 11:15:07
error_code                                        500
error_message                   Internal Server Error
endpoint                            /api/pdf/download


There is an error message pointing to the PDF download endpoint. Can we trace this back to specific users?

### Web Tracking Event Log

This table connects session IDs with user IDs and shows how long users stayed on the site.

In [ ]:
show_row = web_tracking_event_log.iloc[0].to_frame()
print(show_row)

                                                                 0
session_id                    024e70b0-bc9d-4e67-b289-3e4714c59ed3
user_id                       caffcbdc-eabc-47a8-9204-4b98c7e9bdd7
session_start                                  2025-10-16 10:59:54
session_end                                    2025-10-16 11:02:02
session_duration                                               128
landing_page                                              /product
referrer                                                       ads
pdf_download_click_timestamp                                   NaN


The session ID and user ID are linked here. Session duration could be interesting for comparing user behavior before and after the release.

### Zendesk Tickets

In [ ]:
show_row = zendesk_tickets_export.iloc[0].to_frame()
print(show_row)

                                               0
ticket_id   3e327bfe-f886-42c8-ad2a-f1b08b990c1c
user_id     62a2261a-0f99-47e2-872d-912df863ed4c
created_at                   2025-10-19 05:06:42
category                          Produktanfrage
status                                    solved
priority                                    high


### Connecting the Tables

The tables can be joined via `user_id` and `session_id` using `pd.merge()`. However, for the actual analysis I will use SQL, as it is better suited for structured queries across multiple tables.

### Key Questions for the SQL Analysis

- Did error logs increase after the release of the download button?
- How long do customers stay on the website before vs. after the release? Are there differences between users with and without errors?
- Did support tickets actually decrease?
- When exactly do errors occur (OS, browser, channel)?

*Decision: The data was loaded into a Databricks environment for structured SQL querying.*

---
## Part 2: SQL Analysis (Databricks)

All following queries reference the same dataset explored above.

### Setup

In [ ]:
%sql
USE CATALOG saas_project;
USE SCHEMA default;

### Total Tickets Before and After Deployment

Did the overall number of support tickets change after the release?

In [ ]:
%sql
SELECT COUNT(*) AS tickets_before_deployment
FROM zendesk_tickets_export
WHERE created_at < (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_before_deployment
-------------------------
                     1399


In [ ]:
%sql
SELECT COUNT(*) AS tickets_after_deployment
FROM zendesk_tickets_export
WHERE created_at > (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_after_deployment
------------------------
                    1601


### Product Inquiries Before and After Deployment

Filtering for the category "Produktanfrage" — did the release reduce product-related inquiries specifically?

In [ ]:
%sql
SELECT count_if(category = 'Produktanfrage') AS tickets_before_deployment_product
FROM zendesk_tickets_export
WHERE created_at < (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_before_deployment_product
---------------------------------
                              769


In [ ]:
%sql
SELECT count_if(category = 'Produktanfrage') AS tickets_after_deployment_product
FROM zendesk_tickets_export
WHERE created_at > (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_after_deployment_product
--------------------------------
                             661


### Bug Reports Before and After Deployment

Did the release introduce new bugs or resolve existing ones?

In [ ]:
%sql
SELECT count_if(category = 'Bug Report') AS tickets_before_deployment_bugs
FROM zendesk_tickets_export
WHERE created_at < (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_before_deployment_bugs
------------------------------
                           157


In [ ]:
%sql
SELECT count_if(category = 'Bug Report') AS tickets_after_deployment_bugs
FROM zendesk_tickets_export
WHERE created_at > (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

tickets_after_deployment_bugs
-----------------------------
                          257


### Backend Errors Before and After Deployment

Comparing the number of backend errors — did the release cause more or fewer errors?

In [ ]:
%sql
SELECT COUNT(*) AS error_before_deployment
FROM sys_backend_error_logs
WHERE error_timestamp < (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

error_before_deployment
-----------------------
                   3084


In [ ]:
%sql
SELECT COUNT(*) AS error_after_deployment
FROM sys_backend_error_logs
WHERE error_timestamp > (
  SELECT deployed_at FROM deployment_history_logs
  WHERE notes = 'Release v1.2 (PDF flow update)'
);

error_after_deployment
----------------------
                  3590


### Error Categories

Which error messages occur and how frequently?

In [ ]:
%sql
SELECT error_message, COUNT(*) AS count
FROM sys_backend_error_logs
GROUP BY error_message
ORDER BY count DESC;

error_message         | count
----------------------+------
Internal Server Error |  6674


### Affected Endpoints

Which endpoints are affected by Internal Server Errors?

In [ ]:
%sql
SELECT DISTINCT endpoint
FROM sys_backend_error_logs
WHERE error_message = 'Internal Server Error';

endpoint         
-----------------
/api/pdf/download


---
## Next Steps

- Investigate whether errors correlate with specific browsers or operating systems
- Narrow down the exact timing of error occurrences